### Extract Load Transform
This notebook do the following transformation in **Dados Dados Carregamento_24-25-26_RAMPs.xlsx** and **Direção Vento_RAMPs 09 e 10.xlsx**.
- Converty datatype for each columns.
- Filter this data for **non-nan values from column Dados Dados Carregamento_24-25-26_RAMPs.xlsx**.
- Applying dummie encoding to **Produto**.
- Agregating columns **Wind Direction** from **Direção Vento_RAMPs 09 e 10.xlsx** using **Vector Averaging**.
- Getting Weather information.
- Applying MICE.

In [ ]:
import pandas as pd 

#Getting data
df = pd.read_excel(r"/home/jorgemetri/Desktop/git_repositories/Fugitive-Industrial-Emission-Mining-Company/data/raw/Dados Carregamento_24-25-26_RAMPs.xlsx")
df.info()

In [ ]:
#Convert datatype for each column
for col in df.select_dtypes(include=["object"]).columns:
    # Replace whitespace-only strings with NaN
    cleaned = df[col].replace(r'^\s*$', pd.NA, regex=True)
    non_null = cleaned.dropna()

    if len(non_null) == 0:
        continue

    # Numeric if every non-null value is a number
    if pd.to_numeric(non_null, errors="coerce").notna().all():
        df[col] = pd.to_numeric(cleaned, errors="coerce")
        continue

    # Datetime if every non-null value is a date
    if pd.to_datetime(non_null, errors="coerce").notna().all():
        df[col] = pd.to_datetime(cleaned, errors="coerce")
#Renaming column Dia / Hora
df = df.rename(columns={'Dia / Hora':'Data-Hora'})
df.info()

In [ ]:
df.columns

In [ ]:
#Filter dataframe by non-nan values from column "Taxa de Emissão PIER kg/h"
df= df.dropna(subset=['Taxa de Emissão PIER\nkg/h'])
df['Taxa de Emissão PIER\nkg/h']

In [ ]:
#Applying dummie encoding to Produto Column
df = pd.get_dummies(df,columns=['Produto'],dtype=int)
df

In [ ]:
df['Data-Hora'].unique()

In [ ]:
# Getting Weather information.
import pandas as pd
import requests
from typing import Sequence

OPENMETEO_ARCHIVE_URL = "https://archive-api.open-meteo.com/v1/archive"

DEFAULT_HOURLY_VARS = (
    "temperature_2m",
    "relative_humidity_2m",
    "dew_point_2m",
    "pressure_msl",
    "surface_pressure",
    "wind_speed_10m",
    "wind_direction_10m",
    "wind_gusts_10m",
    "precipitation",
    "cloud_cover",
    "shortwave_radiation",
    "boundary_layer_height",
)


def fetch_openmeteo_hourly(
    df: pd.DataFrame,
    latitude: float,
    longitude: float,
    datetime_col: str = "Data-Hora",
    variables: Sequence[str] = DEFAULT_HOURLY_VARS,
    timezone: str = "America/Sao_Paulo",
    timeout: int = 60,
) -> pd.DataFrame:
    """
    Busca dados meteorológicos horários da Open-Meteo (ERA5) cobrindo todo
    o intervalo temporal presente em `df[datetime_col]`.
    """
    times = pd.to_datetime(df[datetime_col])
    if times.dt.tz is not None:
        times = times.dt.tz_convert(timezone).dt.tz_localize(None)

    start_date = times.min().strftime("%Y-%m-%d")
    end_date   = times.max().strftime("%Y-%m-%d")

    params = {
        "latitude": latitude,
        "longitude": longitude,
        "start_date": start_date,
        "end_date": end_date,
        "hourly": ",".join(variables),
        "timezone": timezone,
        "wind_speed_unit": "ms",
        "temperature_unit": "celsius",
        "precipitation_unit": "mm",
    }

    resp = requests.get(OPENMETEO_ARCHIVE_URL, params=params, timeout=timeout)
    resp.raise_for_status()
    payload = resp.json()

    hourly = payload["hourly"]
    weather = pd.DataFrame(hourly)
    weather["time"] = pd.to_datetime(weather["time"])
    weather = (
        weather
        .rename(columns={"time": datetime_col})
        .set_index(datetime_col)
        .sort_index()
    )

    return weather

import numpy as np

fixed_coords = np.array([
    [-40.57039, -20.78534],   # sensor 10
    [-40.57175, -20.78040],   # sensor 11
    [-40.57180, -20.78845],   # sensor 12
])

centroid_lon, centroid_lat = fixed_coords.mean(axis=0)

#Applying the function
wx = fetch_openmeteo_hourly(
    df,
    latitude=centroid_lat,
    longitude=centroid_lon,
    datetime_col="Data-Hora",
)

df= df.merge(wx, left_on="Data-Hora", right_index=True, how="left")

In [ ]:
# fill nan values from Weather Dataframe using xgboosting
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, r2_score


def impute_blh_xgboost(
    df: pd.DataFrame,
    datetime_col: str = "Data-Hora",
    target: str = "boundary_layer_height",
    random_state: int = 42,
    verbose: bool = True,
) -> pd.DataFrame:
    df = df.sort_values(datetime_col).reset_index(drop=True).copy()

    dt = df[datetime_col]
    hour = dt.dt.hour + dt.dt.minute / 60
    doy  = dt.dt.dayofyear

    df["_hour_sin"] = np.sin(2 * np.pi * hour / 24)
    df["_hour_cos"] = np.cos(2 * np.pi * hour / 24)
    df["_doy_sin"]  = np.sin(2 * np.pi * doy / 365.25)
    df["_doy_cos"]  = np.cos(2 * np.pi * doy / 365.25)

    df["_swrad_cumsum_day"] = (
        df.groupby(dt.dt.date)["shortwave_radiation"].cumsum()
    )

    for k in [1, 2, 3, 24]:
        df[f"_{target}_lag{k}"]  = df[target].shift(k)
        df[f"_{target}_lead{k}"] = df[target].shift(-k)

    df["_t_minus_td"] = df["temperature_2m"] - df["dew_point_2m"]

    feature_cols = [
        "temperature_2m", "relative_humidity_2m", "dew_point_2m",
        "pressure_msl", "surface_pressure",
        "wind_speed_10m", "wind_direction_10m", "wind_gusts_10m",
        "precipitation", "cloud_cover", "shortwave_radiation",
        "_hour_sin", "_hour_cos", "_doy_sin", "_doy_cos",
        "_swrad_cumsum_day", "_t_minus_td",
        f"_{target}_lag1", f"_{target}_lag2", f"_{target}_lag3", f"_{target}_lag24",
        f"_{target}_lead1", f"_{target}_lead2", f"_{target}_lead3", f"_{target}_lead24",
    ]

    mask_known = df[target].notna()
    mask_impute = df[target].isna()

    X_train = df.loc[mask_known, feature_cols]
    y_train = df.loc[mask_known, target]
    X_pred  = df.loc[mask_impute, feature_cols]

    model_params = dict(
        n_estimators=2000,
        learning_rate=0.03,
        max_depth=6,
        min_child_weight=5,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_lambda=1.0,
        tree_method="hist",
        early_stopping_rounds=50,
        random_state=random_state,
    )

    if verbose:
        kf = KFold(n_splits=5, shuffle=True, random_state=random_state)
        rmses, r2s = [], []
        for tr_idx, va_idx in kf.split(X_train):
            m = xgb.XGBRegressor(**model_params)
            m.fit(
                X_train.iloc[tr_idx], y_train.iloc[tr_idx],
                eval_set=[(X_train.iloc[va_idx], y_train.iloc[va_idx])],
                verbose=False,
            )
            pred = m.predict(X_train.iloc[va_idx])
            rmses.append(np.sqrt(mean_squared_error(y_train.iloc[va_idx], pred)))
            r2s.append(r2_score(y_train.iloc[va_idx], pred))
        print(f"[CV 5-fold]  RMSE: {np.mean(rmses):7.1f} ± {np.std(rmses):.1f} m")
        print(f"[CV 5-fold]  R²:   {np.mean(r2s):.4f} ± {np.std(r2s):.4f}")
        print(f"[baseline]   σ(y): {y_train.std():7.1f} m   (compare com RMSE acima)")

    n = len(X_train)
    rng = np.random.default_rng(random_state)
    val_idx = rng.choice(n, size=n // 10, replace=False)
    tr_mask = np.ones(n, dtype=bool); tr_mask[val_idx] = False

    final_model = xgb.XGBRegressor(**model_params)
    final_model.fit(
        X_train.iloc[tr_mask], y_train.iloc[tr_mask],
        eval_set=[(X_train.iloc[~tr_mask], y_train.iloc[~tr_mask])],
        verbose=False,
    )

    df.loc[mask_impute, target] = final_model.predict(X_pred)
    df[f"{target}_imputed"] = mask_impute.values

    df = df.drop(columns=[c for c in df.columns if c.startswith("_")])

    if verbose:
        print(f"\n[impute]     {mask_impute.sum()} linhas imputadas de {len(df)} totais ({100*mask_impute.mean():.1f}%)")

    return df


df = impute_blh_xgboost(df)

### Generating processed .csv

In [ ]:
df

In [ ]:
#Save processed dataframe as .csv
output_path = r"/home/jorgemetri/Desktop/git_repositories/Fugitive-Industrial-Emission-Mining-Company/data/processed/Dataset.csv"
df.to_csv(output_path, index=False)
print(f"Saved: {output_path}")